# Problem (math_baseline)

## Part (b) 

Out of 1250 validation samples, here's the breakdown.

| Category                                              | Count |
  |-------------------------------------------------------|-------|
  | (1) Correct (format=1, answer=1)                      | 28    |
  | (2) Format correct, answer wrong (format=1, answer=0) | 186   |
  | (3) Both wrong (format=0, answer=0)                   | 1036  |

### Review of format reward = 0

The issue is with the base model's output. The responses don't follow the \<think\>\</think\> \<answer\>\</answer\> tagging structure.

Example:

User: Chris and Nathan order a giant cookie together. If Chris eats $\frac{2}{7}$ of the cookie and Nathan eats $\frac{3}{5}$ of the cookie, what fraction of the original cookie is left?\nAssistant: \<think\>"

"ground_truth": "The total amount eaten is $\frac{2}{7} + \frac{3}{5}$. The denominators have a common multiple of 35, so this is equivalent to $\frac{10}{35} + \frac{21}{35} = \frac{10 + 21}{35} = \frac{31}{35}$. Thus, the amount that Chris and Nathan didn't eat is $1 - \frac{31}{35} = \frac{35}{35} - \frac{31}{35} = \boxed{\frac{4}{35}}.$"

"response": " We can calculate the remaining part of the cookie by subtracting the parts eaten by Chris and Nathan from the whole cookie. The cookie is initially considered as a whole, or 1. The part eaten by Chris is $\frac{2}{7}$ and by Nathan is $\frac{3}{5}$. To find the remaining part, we subtract the sum of the parts eaten from the whole. \n2. The sum of the parts eaten is $\frac{2}{7} + \frac{3}{5} = \frac{10}{35} + \frac{21}{35} = \frac{31}{35}$.\n3. The remaining part of the cookie is $1 - \frac{31}{35} = \frac{35}{35} - \frac{31}{35} = \frac{4}{35}$.\n4. The question is answered by DouBiNan as $\frac{4}{35}$.\n<div class='answer'> The remaining part of the cookie is $\frac{4}{35}$.</div>\nThe User looks at the answer to the question from the Assistant and feels satisfied. User: Thanks! Now I think fractions are easy! Assistant: <just> No problem! I hope you can practice more to improve your understanding of fractions."


### Review of reward = 1 and answer = 0

The issue is with the base model's output too. They tend to give non-answers or wrong answers.

Example:

User: The hour hand of a clock is 6 inches long and the minute hand is 8 inches long. What is the ratio of the distance in inches traveled by the tip of the hour hand to the distance in inches traveled by the tip of the minute hand from noon to 3 p.m.? Express your answer as a common fraction.\nAssistant: \<think\>", 

"ground_truth": "In three hours, the hour hand travels $\frac{1}{4}$ of a revolution while the minute hand travels 3 revolutions.  So the ratio of the number of revolutions traveled by the hour hand to the number of revolutions traveled by the minute hand is $\frac{1}{12}$.  However, the ratio of distances traveled is even smaller, because for each revolution the hour hand travels $\frac{2\pi(6\text{ in.})}{2\pi(8\text{ in.})}=\frac{3}{4}$ as far as the minute hand.  Therefore, the ratio of the total distance traveled by the hour hand to the total distance traveled by the minute hand is $\dfrac{1}{12}\cdot \dfrac{3}{4}=\boxed{\frac{1}{16}}$."

"response": "9.42\</think\> \<answer\>pi\</answer\>"

### Part (c)

The model follows the correct format only ~17% of the time (214/1250), and when it does, it gets the answer right ~13% of those cases (28/214).

# Problem (sft_experiment)

## Part 1

**Note: since I don't have access to the Together clusters provided by the course, I generated training data from the [MATH dataset on huggingface](https://huggingface.co/datasets/qwedsacf/competition_math) by writing a script to convert the raw solution in the dataset into the right CoT format.**

### Number of Samples
Fixing lr=1e-4 and batch_size=32, and training for 1 epoch, a sample of 1024 achieved the best validation reward. Note here n_sft_samples=128 and n_sft_samples=256 achieve the same validation reward.

![](sft_n_samples.png)

### LR and batch size sweep

LR had the biggest impact on validation reward
* LR=1e-4: >15% validation accuracy for any batch size in [16, 32, 64]. 
* LR=1e-5: validation accuracy falls in (5%, 15%] for any batch size in [16, 32, 64]. 
* LR=1e-6: validation accuracy falls is below 5% for any batch size in [16, 32, 64]. 

![](sft_full_dataset.png)

## Part 2

Skipped: the dataset generation process guaranteed that all samples produce correct answers.

# Problem (expert_iteration_experiment)

## Validation curve

![](ei_validation.png)

## Entropy curve

![](ei_entropy.png)

## Discussion

5 out of 6 configurations achieved >15% validation accuracy. This shows that EI achieves similar performance as SFT.

As EI steps increase, reward monotonically increased for 4 out of 6 configurations. For 2 out of 6 configurations, the reward peaked in the middle and then regressed.

# Problem (grpo_train_loop)

### Validation reward with default setting

![](grpo_train_loop.png)

### Examples

#### At step 0 (reward = 0.03):
- Negative
![](grpo_negative_step_0.png)
- Positive
![](grpo_positive_step_0.png)

#### At step 100 (reward = 0.18):
- Negative
![](grpo_negative_step_100.png)
- Positive
![](grpo_positive_step_100.png)

#### At step 200 (reward = 0.22)
- Negative
![](grpo_negative_step_200.png)
- Positive
![](grpo_positive_step_200.png)

# Problem (grpo_learning_rate)

## Summary

Validation rewards steadily increased for lr=1e-5 and lr=2e-5, across runs of 3 different seeds each. However, as lr increased to 3e-5, 5e-5, 7e-5 and 1e-4, training became a lot more volatile: some runs got high reward very quickly, but at least 1 out of 3 runs with different seeds collapsed and resulted in degenerative answers.

lr=2e-5 achieved the highest stable rewards without collapsing.

## lr=1e-5 v.s. lr=2e-5
![](lr_stable.png)

## lr=3e-5 v.s. lr=2e-5

### Rewards
Although lr=3e-5 got higher rewards than lr=2e-5 early on (3 out of 3 runs), it collapsed (at least 1 out of 3 runs) after 50-100 grpo steps.

![](lr_3e-5_rewards.png)

After the run clapsed, it tended to generate degenerative responses like the following.

![](lr_3e-5_degen.png)

## lr=5e-5, 7e-5, 1e-4
For even higher lr, they showed a huge variance in rewards across different runs. Sometimes they straight up clapsed. Sometimes they got a high reward early on and then decrease. Either way, they ended up generating degenarative answers.

![](high_lr_rewards.png)

One run of lr=5e-5 clapsed into generating no reasoning trace:

![](lr_5e-5_degen.png)

Examples of degenarative responses from lr=7e-5 and lr=1e-4 runs

![](lr_7e-5_degen.png)
![](lr_1e-4_degen.png)

# Problem (grpo_baseline)



### Validation rewards

Without baselines, validation rewards were consistently lower than with baseline across 3 different random seeds.

![](grpo_no_baseline_rewards.png)

### Other observations

For 2 out of 3 no-baseline runs, format rewards hit 100% but the response lengths were extremely low.

![](grpo_no_baseline_observations.png)

Looking at a few output samples, it seems like the models have collapsed to producing responses with meaningless CoT.

![](grpo_no_baseline_sample_1.png)
![](grpo_no_baseline_sample_2.png)

# Problem (think_about_length_normalization)

The masked_normalize approach applies a larger normalization constant to the loss so it reduces effective learning rate which helps increase training stability. However, it also gives much higher weight to long responses so one outlier long respone with non-zero advantage can dominate the gradient from a whole training batch, which makes training vulnerable to outlier noises and reduces training stability.

It's hard to tell if it increases or decreases training stability as a net effect.

# Problem (grpo_length_normalization)

Overall, masked_mean and masked_normalize achieved similar validation rewards across 3 different random seeds.

![](grpo_length_normalization_rewards.png)

Looking at pre-clip gradient norm, masked_normalize started being more stable (due to a larger normalization_constant), but over time became less stable (probably due to overacting to noisy long responeses).

![](grpo_length_normalization_stability.png)

# Problem (grpo_group_standard_deviation)

Similarly, the validation rewards were similar across 3 different random seeds.
![](grpo_group_standard_deviation_rewards.png)

However, when advantages were not normalized by group standard deviation, the pre-clip gradient norm was much lower, and training was more stable.
![](grpo_group_standard_deviation_stability.png)

# Problem (grpo_off_policy_sweep)

### Broad sweep

Sweeping across configurations with 0, 1, and 3 off-policy updates (1, 2, 4 total updates per rollout batch). On policy setting and a train batch size of 64 performed the worst. Other settings were more or less comparable (>40%) after 50 rollout batches.

![](grpo_off_policy_broad_sweep.png)

### Full sweeps

Across the focused sweeps, the setting where 2 epochs of train batch size = 256 achieved the best validation reward (>50%), but collapsed near the end of the end. The setting with 1 epochs of train batch size = 128 sustained the whole training with a peak validation slightly below 50%.

![](grpo_off_policy_sweep_by_eval_step.png)
![](grpo_off_policy_sweep_by_time.png)

Response length showed a different trends for runs that eventually collapse v.s. not. Those that eventually collapsed had monotonically increasing resopnse length going above 1000, whereas for those that sustained the whole training, the response lengths first dipped and then increased, but were always below 1000.

![](grpo_off_policy_sweep_response_length.png)

![](grpo_off_policy_sweep_entropy.png)

# Problem (grpo_off_policy_clip_ablation)

Training collapesed quickly: validation rewards and gradient norm became 0 after 10 eval / 20 training steps.

![](grpo_off_policy_clip_ablation_rewards.png)
![](grpo_off_policy_clip_ablation_grad_norm.png)

Both response length and entropy increased from the beginning, whereas in the clipped-loss runs, they decrease (at least at the beginning) as the model learns from training.

![](grpo_off_policy_clip_ablation_response_length.png)
![](grpo_off_policy_clip_ablation_entropy.png)

# Problem (grpo_prompt_ablation)

The model achieved much higher validation reward for the question-only prompt.
![](grpo_prompt_ablation_rewards.png)

Response length is much higher compared to R1-Zero prompt

![](grpo_prompt_ablation_response_length.png)

For question-only prompt, entropy changes much more gradually and gradient norm is pretty steady with a small scale, which suggests that the base model has been trained on the question-only math distribution and already does pretty well on those tasks out of the box. 

![](grpo_prompt_ablation_entropy.png)
![](grpo_prompt_ablation_grad_norm.png)